# Buổi 30 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `kien_truc.py`; các ô tự dùng bản mới.
Cả notebook chạy khoảng 10 phút trên CPU 4 nhân.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Nhu cầu điện 5 vùng và nhiệt độ (mục 4.1)

In [ ]:
%matplotlib inline
import warnings
from functools import reduce

import kien_truc as kt

warnings.simplefilter("ignore")
d = kt.doc_du_lieu()
moc = kt.cac_moc_test()
print(d.groupby("unique_id").size().to_dict(), "giờ mỗi vùng;", d.attrs["so_lo_i"], "giờ nhu cầu lỗi đã lấp")
print("mốc test:", moc[0].date(), "→", moc[-1].date(), f"({len(moc)} mốc)")
d[d["unique_id"] == "PJM"].iloc[-3:][["ds", "y", "eia_du_bao", "nhiet_do_thuc", "nhiet_do_du_bao", "gio", "thu"]]

## Bước 2 — Ba loại covariate (mục 4.1)

In [ ]:
print("FUTR_EXOG:", kt.FUTR_EXOG, "| HIST_EXOG:", kt.HIST_EXOG, "| STAT_EXOG:", kt.STAT_EXOG)
m0 = kt.MOC_TEST
_, futr = kt.du_lieu_moc(d, m0)
print(futr.head(3))

## Bước 3 — Năm kiến trúc và các baseline (mục 4.2–4.6) — khoảng 7 phút

In [ ]:
nf, giay, bang = {}, {}, [kt.seasonal_naive(d, moc), kt.eia(d, moc), kt.mstl(d, moc), kt.lightgbm(d, moc)]
for ten in kt.KIEN_TRUC:
    nf[ten], giay[ten] = kt.huan_luyen(d, ten)
    f = kt.du_bao_cac_moc(nf[ten], d, moc)
    if ten == "DeepAR":
        khoang = f.copy()                      # giữ cận khoảng cho bước 6
        f = f.drop(columns=["DeepAR"]).rename(columns={"DeepAR-median": "DeepAR"})
    bang.append(f[["unique_id", "ds", "moc", ten]])
    print(f"{ten:8s} huấn luyện {giay[ten]:6.1f} giây")
x = reduce(lambda a, b: a.merge(b, on=["unique_id", "ds", "moc"]), bang)
kt.bang_mase(x, d, [c for c in x.columns if c not in ("unique_id", "ds", "moc")])

## Bước 4 — Cái giá của rò rỉ `futr_exog` (mục 4.1) — khoảng 1 phút

In [ ]:
nf_ro_ri, _ = kt.huan_luyen(d, "NHITS", futr=["nhiet_do_thuc", "gio", "thu"], hist=[])
a = kt.du_bao_cac_moc(nf_ro_ri, d, moc).rename(columns={"NHITS": "rò rỉ: backtest"})
b = kt.du_bao_cac_moc(nf_ro_ri, d, moc, thay_futr={"nhiet_do_thuc": "nhiet_do_du_bao"}).rename(columns={"NHITS": "rò rỉ: vận hành"})
c = x[["unique_id", "ds", "moc", "NHITS"]].rename(columns={"NHITS": "khai đúng"})
r = reduce(lambda u, v: u.merge(v, on=["unique_id", "ds", "moc"]), [a, b, c])
kt.bang_mase(r, d, ["rò rỉ: backtest", "rò rỉ: vận hành", "khai đúng"])

## Bước 5 — Nhìn vào trong N-BEATS và TFT (mục 4.2, 4.4)

In [ ]:
tp = kt.thanh_phan_nbeats(nf["NBEATS"], d, moc[20], "ERCO")
print(tp.round(0).iloc[[0, 6, 12, 18, 23]])
for loai, w in kt.tft_chon_bien(nf["TFT"]).items():
    print(loai, w.round(3).to_dict())

## Bước 6 — Khoảng dự báo của DeepAR có đúng 80% không (mục 4.3)

In [ ]:
cv = kt.coverage_theo_tam(khoang, d, 80)
print("phủ chung:", round(cv.mean(), 3))
cv.round(3).iloc[[0, 5, 11, 17, 23]]

## Bước 7 — Kiểm tra

Trong terminal ở thư mục `lab/`: `python lab.py check` → 5/5 xanh.